In [1]:
import numpy as np
import matplotlib.pyplot as plt

class NeuralNetworkWithRegularization:
    def __init__(self, input_size, hidden_size, output_size, reg_type='l2', reg_strength=0.01, dropout_p=0.5):
        # He初始化权重
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros(output_size)
        
        self.reg_type = reg_type
        self.reg_strength = reg_strength
        self.dropout_p = dropout_p
        self.dropout_mask = None
    
    def regularization_loss(self):
        """计算正则化损失"""
        if self.reg_type == 'l2':
            # L2正则化
            return 0.5 * self.reg_strength * (np.sum(self.W1**2) + np.sum(self.W2**2))
        elif self.reg_type == 'l1':
            # L1正则化
            return self.reg_strength * (np.sum(np.abs(self.W1)) + np.sum(np.abs(self.W2)))
        else:
            return 0
    
    def regularization_gradient(self, W):
        """计算正则化梯度"""
        if self.reg_type == 'l2':
            return self.reg_strength * W
        elif self.reg_type == 'l1':
            return self.reg_strength * np.sign(W)
        else:
            return 0
    
    def forward_train(self, X):
        """训练时的前向传播（带dropout）"""
        # 第一层
        z1 = np.dot(X, self.W1) + self.b1
        a1 = np.maximum(0, z1)  # ReLU
        
        # Dropout
        self.dropout_mask = (np.random.rand(*a1.shape) < self.dropout_p) / self.dropout_p
        a1 = a1 * self.dropout_mask
        
        # 第二层
        z2 = np.dot(a1, self.W2) + self.b2
        a2 = np.maximum(0, z2)
        
        return a2
    
    def forward_test(self, X):
        """测试时的前向传播（无dropout）"""
        z1 = np.dot(X, self.W1) + self.b1
        a1 = np.maximum(0, z1)
        
        # 测试时不做dropout，但需要缩放（因为训练时除以了p）
        # 这里使用inverted dropout，训练时已经缩放，测试时不需要
        # 所以直接使用即可
        
        z2 = np.dot(a1, self.W2) + self.b2
        a2 = np.maximum(0, z2)
        
        return a2